In [2]:
import GEOparse

gse_tumor = GEOparse.get_GEO("GSE4271", destdir="../data/raw/geo",silent=True)
gse_normal = GEOparse.get_GEO("GSE6735", destdir="../data/raw/geo",silent=True)

In [3]:
import pandas as pd
import numpy as np

def build_expr_matrix(gse):
    exprs = []

    for gsm_name, gsm in gse.gsms.items():
        table = gsm.table

        expr = table[["ID_REF", "VALUE"]].copy()
        expr.columns = ["probe", gsm_name]

        exprs.append(expr)

    merged = exprs[0]

    for e in exprs[1:]:
        merged = merged.merge(e, on="probe")

    return merged.set_index("probe")

tumor_expr = build_expr_matrix(gse_tumor)
normal_expr = build_expr_matrix(gse_normal)

In [4]:
tumor_expr = np.log2(tumor_expr + 1)
normal_expr = np.log2(normal_expr + 1)

normal_expr["row_mean"] = normal_expr.mean(axis=1)
normal_expr = normal_expr["row_mean"]

In [9]:
print("Tumor max:",tumor_expr.to_numpy().max())
print("Tumor min:",tumor_expr.to_numpy().min())
print("Healthy:", normal_expr.describe().T[['min','max']])

Tumor max: 17.731069057150133
Tumor min: 0.765534746362977
Healthy: min     0.431753
max    13.052201
Name: row_mean, dtype: float64
